<a href="https://colab.research.google.com/github/leticiasdrummond/Modelos-Base/blob/main/An%C3%A1lise_e_Corre%C3%A7%C3%A3o_de_Modelo_de_Microrrede.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Esta análise identifica as divergências entre o código original ("Caso Referência 1.2.1") e a fundamentação teórica contida no anexo técnico (NAGATA et al., 2023), propondo as correções necessárias para garantir o rigor acadêmico e a precisão dos resultados.

### 1. Comparação e Identificação de Divergências

Ao comparar o arquivo `1.2.1 Caso Referencia - Min Custo Energ.ipynb` com o artigo `Otimização do Gerenciamento de Microrrede para Recarga de Veículos Elétricos`, observam-se as seguintes discrepâncias:

| Componente | Código Original (1.2.1) | Modelo Proposto (Anexo PDF) | Impacto da Divergência |
| --- | --- | --- | --- |
| **Balanço de Potência** | $G_{pv} + P_{buy} = D_{com} + D_{ev} + P_{sell} + P_{bess}$ | $P_t^S + P_t^{GD} + P_t^{AEe} = P_t^c + P_t^V + P_t^{AE}$ | O código original usa variáveis simplificadas. A adoção da nomenclatura do PDF ($P_t^S$ para compra, $P_t^V$ para venda) padroniza a citação. |
| **Dinâmica do BESS** | $SOC_t = SOC_{t-1} + (P_c \cdot \eta - P_d / \eta)$ | $E_t^{AE} = E_{t-1}^{AE} \cdot \beta + (P_t^{AE} \cdot \alpha - P_t^{AEe} / \alpha) \cdot \delta$ | O código ignora a **taxa de autodescarga ($\beta$)** e o **intervalo de tempo ($\delta$)**, o que superestima a energia disponível em horizontes longos. |
| **Degradação** | Custo fixo por kWh (Throughput) | Não explicitado na F.O. principal do PDF | Embora o custo de degradação seja uma boa prática, para fidelidade ao anexo, ele deve ser ajustado ou justificado como extensão. |
| **Eficiência ($\alpha$)** | 0.955 (95.5%) | 0.90 (90%) | O código utiliza uma eficiência muito otimista. O PDF sugere 90%, resultando em perdas de conversão maiores. |

### 2. Análise de Impacto das Modificações

As modificações propostas visam aproximar o modelo da realidade física descrita por Nagata et al. (2023):

* **Ajuste da Eficiência ($\alpha=0.9$):** Reduzirá a atratividade do uso da bateria para pequenas diferenças de preço, pois as perdas no ciclo de carga/descarga serão maiores.
* **Inclusão da Autodescarga ($\beta$):** Fará com que o sistema evite manter a bateria carregada por períodos ociosos muito longos sem necessidade.
* **Padronização Simbólica:** Facilita a validação dos resultados em relação às figuras e tabelas do artigo citado.

---

### 3. Código Corrigido (Modelo Concreto Pyomo)

In [4]:
import pyomo.environ as pyo
import pandas as pd
import numpy as np

# ============================================================
# 1. CARACTERIZAÇÃO: MODELO CONCRETO (NAGATA et al., 2023)
# ============================================================
model = pyo.ConcreteModel(name="Minimizacao_Custo_Energia_EV")

# ============================================================
# 2. DADOS DE ENTRADA (REFERÊNCIA ABNT: NAGATA et al., 2023)
# ============================================================
HORIZONTE = 24
T = range(HORIZONTE)

# Parâmetros conforme Tabela 1 e Quadro 1 do PDF
delta = 1.0        # Intervalo de tempo (1 hora)
beta = 0.99        # Taxa de autodescarga (ajustada conforme literatura base do PDF)
alpha = 0.90       # Eficiência do banco de baterias (α)
cap_max = 50.0     # Capacidade nominal (kWh)
p_ae_max = 15.0    # Potência máxima de injeção/extração (kW)

custo_compra = 0.75  # c_s (R$/kWh)
preco_venda = 0.40   # c_v (R$/kWh)

# Perfis de Carga e Geração (conforme 1.2.1 Caso Referência)
demanda_total = [14, 14, 14, 14, 17, 21, 28, 35, 49, 56, 56, 49,
                 119, 141, 114, 40, 40, 35, 106, 121, 90, 12, 14, 14] # Comércio + EV
geracao_gd = [0, 0, 0, 0, 0, 0, 2, 12, 28, 42, 48, 50,
              50, 48, 42, 28, 12, 2, 0, 0, 0, 0, 0, 0] # P_GD

# ============================================================
# 3. VARIÁVEIS DE DECISÃO
# ============================================================
model.Pt_S = pyo.Var(T, domain=pyo.NonNegativeReals) # Compra da rede
model.Pt_V = pyo.Var(T, domain=pyo.NonNegativeReals) # Venda para rede
model.Pt_AE = pyo.Var(T, bounds=(0, p_ae_max))       # Carga (Injeção)
model.Pt_AEe = pyo.Var(T, bounds=(0, p_ae_max))      # Descarga (Extração)
model.Et_AE = pyo.Var(T, bounds=(0.2*cap_max, 0.95*cap_max)) # Energia (SOC)

# ============================================================
# 4. FUNÇÃO OBJETIVO
# ============================================================
def obj_rule(model):
    return sum(model.Pt_S[t] * custo_compra - model.Pt_V[t] * preco_venda for t in T)
model.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

# ============================================================
# 5. RESTRIÇÕES (FORMULAÇÃO CONFORME NAGATA et al., 2023)
# ============================================================

# Balanço de Potência: GD + Compra + Descarga = Demanda + Venda + Carga
def balance_rule(model, t):
    return geracao_gd[t] + model.Pt_S[t] + model.Pt_AEe[t] == demanda_total[t] + model.Pt_V[t] + model.Pt_AE[t]
model.res_balanco = pyo.Constraint(T, rule=balance_rule)

# Dinâmica da Bateria: Inclui autodescarga (beta) e eficiência (alpha)
def battery_dynamics(model, t):
    if t == 0:
        return model.Et_AE[t] == (0.5 * cap_max) * beta + (model.Pt_AE[t] * alpha - model.Pt_AEe[t] / alpha) * delta
    return model.Et_AE[t] == model.Et_AE[t-1] * beta + (model.Pt_AE[t] * alpha - model.Pt_AEe[t] / alpha) * delta
model.res_bateria = pyo.Constraint(T, rule=battery_dynamics)

# ============================================================
# 6. RESOLUÇÃO
# ============================================================
solver = pyo.SolverFactory('cbc')


### Referência Bibliográfica (ABNT)

NAGATA, Erick Akio; FLORES, Marcos Julio Rider; CARMELITO, Bruno Eduardo. **Otimização do Gerenciamento de Microrrede para Recarga de Veículos Elétricos**. In: VIII CONGRESSO BRASILEIRO DE GERAÇÃO DISTRIBUÍDA (CBGD), 2023, Belo Horizonte, MG. Anais... Belo Horizonte, 2023.